# Auto-Detection of Fluorescence Correction Factor

**Goal (QEP-048):** Instead of requiring a user-supplied `correction_factor` for
`FluorescenceCorrectionProcessor`, can we automatically estimate the optimal
correction factor by jointly fitting an ESR model and the correction amplitude
to the "strongest" pixel?

## Approach

The current correction subtracts a scaled version of the mean spectrum:

```
corrected(f) = raw(f) − α · mean_corrected(f)
```

We hypothesize that the raw pixel spectrum can be decomposed as:

```
pixel(f) = ESR(f; θ) + α · mean_corrected(f)
```

We test two variants:
1. **Joint fit**: fit ESR params `θ` and `α` simultaneously via `scipy.optimize.curve_fit`
2. **Profile likelihood**: sweep `α` over [0, 1], fit ESR for each value, find `α` that minimizes residuals

## Data

MIL2_FOV1 — full 1200×1920 measurement, 4×4-binned to 300×480 for speed.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit

from qdmpy.fitting.models import esr14n
from qdmpy.odmr.data import ODMRData
from qdmpy.odmr.io import MatlabLoader
from qdmpy.odmr.processors import (
    BinningProcessor,
    NormalizationProcessor,
    analyze_fluorescence_effects,
)

DATA_FOLDER = Path.home() / 'git' / 'qdmpy' / 'tests' / 'data' / 'MIL2_FOV1'

## 1. Load data

In [ ]:
loader = MatlabLoader(str(DATA_FOLDER))
data_raw = ODMRData.from_loader(loader)
data_binned = BinningProcessor(bin_factor=4).process(data_raw)
data = NormalizationProcessor().process(data_binned)

n_pol = data.data.sizes['polarity']
n_fr  = data.data.sizes['freq_range']
n_y   = data.data.sizes['y']
n_x   = data.data.sizes['x']
n_freq = data.data.sizes['freq_idx']
freq_ghz = data.data.coords['freq_ghz'].values  # (n_fr, n_freq)

for _fr in range(n_fr):
    pass

## 2. Characterise the fluorescence contribution

Before attempting to estimate α, we need to understand how large the fluorescence
contribution actually is in this dataset.

In [ ]:
# Off-resonance baseline per pixel (first + last 4 frequencies)
flat_norm = data.data.values.reshape(n_pol, n_fr, n_y * n_x, n_freq)
off_res = np.concatenate([flat_norm[:, :, :, :4], flat_norm[:, :, :, -4:]], axis=-1)
pixel_baselines = off_res.mean(axis=-1)  # (n_pol, n_fr, n_pix)

# Dip depths after normalization
dip_depth = 1 - flat_norm.min(axis=-1)  # (n_pol, n_fr, n_pix)


In [ ]:
# Spatial map of dip depth and baseline
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

im0 = axes[0].imshow(pixel_baselines[0, 1].reshape(n_y, n_x), cmap='viridis')
axes[0].set_title('Off-resonance baseline (pol=0, fr=1)')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(dip_depth[0, 1].reshape(n_y, n_x), cmap='plasma')
axes[1].set_title('ESR dip depth (pol=0, fr=1)')
plt.colorbar(im1, ax=axes[1])

plt.suptitle('Spatial variation — fluorescence effect is <0.6% baseline shift')
plt.tight_layout()
plt.show()

## 3. Identify the "strong pixel" and mean correction

`analyze_fluorescence_effects` selects the pixel with largest deviation from the spatial
mean and returns the baseline-corrected mean spectrum.

In [ ]:
flat_idx, mean_corr = analyze_fluorescence_effects(data)
y_idx = flat_idx // n_x
x_idx = flat_idx % n_x

# Visualise the strong pixel vs mean for one slice (pol=0, fr=1, high branch)
POL, FR = 0, 1
pixel_spec = data.data.isel(polarity=POL, freq_range=FR, y=y_idx, x=x_idx).values
mean_c     = mean_corr.isel(polarity=POL, freq_range=FR).values
freqs      = freq_ghz[FR]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: actual spectra
mean_spec = data.data.isel(polarity=POL, freq_range=FR).mean(dim=('y', 'x')).values
axes[0].plot(freqs, pixel_spec, 'b.-', lw=1.5, label='Strong pixel')
axes[0].plot(freqs, mean_spec, 'k--', lw=1.5, label='Spatial mean')
axes[0].set_xlabel('Frequency (GHz)')
axes[0].set_ylabel('ODMR contrast')
axes[0].set_title('Strong pixel vs spatial mean')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Right: mean_corrected (the correction template)
axes[1].plot(freqs, mean_c, 'r.-', lw=1.5)
axes[1].set_xlabel('Frequency (GHz)')
axes[1].set_ylabel('Deviation from edge baseline')
axes[1].set_title('mean_corrected(f) — the correction template')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 4. Method A: Joint fit (ESR params + α simultaneously)

Model: `pixel(f) = ESR14N(f; center, width, c0, c1, c2, offset) + α · mean_corrected(f)`

Fit all 7 parameters with `scipy.optimize.curve_fit`.

In [ ]:
def make_joint_model(mean_correction_1d: np.ndarray):
    """Return a composite ESR14N + fluorescence model for scipy.optimize."""
    def model(f, center, width, c0, c1, c2, offset, alpha):
        params = np.array([[center, width, c0, c1, c2, offset]])
        esr = esr14n(f, params).squeeze()
        return esr + alpha * mean_correction_1d
    return model


def run_joint_fit(pixel_spec, mean_c, freqs):
    """Run the joint fit and return (popt, alpha, residual_rms)."""
    composite = make_joint_model(mean_c)
    p0 = [freqs.mean(), 0.002, 0.02, 0.04, 0.02, 0.0, 0.2]
    bounds = (
        [freqs.min(), 1e-4, 0,    0,    0,    -0.2, 0  ],
        [freqs.max(), 0.05, 0.5,  0.5,  0.5,   0.2, 1.0],
    )
    popt, _ = curve_fit(composite, freqs, pixel_spec, p0=p0, bounds=bounds, maxfev=10000)
    fitted = composite(freqs, *popt)
    rms = np.sqrt(np.mean((pixel_spec - fitted) ** 2))
    return popt, popt[-1], rms


popt_real, alpha_joint_real, rms_real = run_joint_fit(pixel_spec, mean_c, freqs)

## 5. Method B: Profile likelihood

For each candidate α ∈ [0, 1], subtract α·mean_corrected from the pixel spectrum,
then fit the ESR model to the residual. The α with the lowest chi² is the estimate.

This avoids simultaneous degeneracy between α and ESR contrast parameters.

In [ ]:
def fit_esr_chi2(alpha, pixel_spec, mean_c, freqs):
    """Subtract alpha*mean_c, fit ESR14N, return sum-of-squared residuals."""
    corrected = pixel_spec - alpha * mean_c

    def esr_model(f, center, width, c0, c1, c2, offset):
        params = np.array([[center, width, c0, c1, c2, offset]])
        return esr14n(f, params).squeeze()

    p0 = [freqs.mean(), 0.002, 0.02, 0.04, 0.02, 0.0]
    bounds = (
        [freqs.min(), 1e-4, 0,    0,    0,    -0.2],
        [freqs.max(), 0.05, 0.5,  0.5,  0.5,   0.2],
    )
    try:
        popt, _ = curve_fit(esr_model, freqs, corrected, p0=p0, bounds=bounds, maxfev=5000)
        return float(np.sum((corrected - esr_model(freqs, *popt)) ** 2))
    except Exception:
        return np.inf


def profile_alpha(pixel_spec, mean_c, freqs, n_steps=60):
    """Sweep alpha over [0, 1], return (alphas, chi2s, best_alpha)."""
    alphas = np.linspace(0.0, 1.0, n_steps)
    chi2s = np.array([fit_esr_chi2(a, pixel_spec, mean_c, freqs) for a in alphas])
    best_alpha = alphas[np.argmin(chi2s)]
    return alphas, chi2s, best_alpha


alphas_real, chi2s_real, alpha_profile_real = profile_alpha(pixel_spec, mean_c, freqs)

plt.figure(figsize=(7, 4))
plt.plot(alphas_real, chi2s_real, 'b.-')
plt.axvline(alpha_profile_real, color='r', ls='--', label=f'best α = {alpha_profile_real:.3f}')
plt.axvline(0.2, color='gray', ls=':', label='default α = 0.2')
plt.xlabel('α (correction factor)')
plt.ylabel('Fit chi² (sum of squared residuals)')
plt.title('Profile likelihood — real MIL2_FOV1 data')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Synthetic validation: known α recovery

We add a controlled fluorescence contribution with known α to the normalized data,
then verify both methods recover it.

In [ ]:
alpha_true_values = [0.0, 0.1, 0.2, 0.35, 0.5, 0.7]

results = []
for alpha_true in alpha_true_values:
    # Inject fluorescence: pixel + alpha_true * mean_corrected
    synth_spec = pixel_spec + alpha_true * mean_c

    # Method A: joint fit
    try:
        _, alpha_joint, rms_j = run_joint_fit(synth_spec, mean_c, freqs)
    except Exception:
        alpha_joint, rms_j = np.nan, np.nan

    # Method B: profile likelihood
    _, _, alpha_profile = profile_alpha(synth_spec, mean_c, freqs, n_steps=80)

    results.append((alpha_true, alpha_joint, alpha_profile))

results = np.array(results)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect recovery')
ax.plot(results[:, 0], results[:, 1], 'bs-', ms=8, label='Joint fit')
ax.plot(results[:, 0], results[:, 2], 'r^-', ms=8, label='Profile likelihood')
ax.set_xlabel('α true')
ax.set_ylabel('α recovered')
ax.set_title('Synthetic validation: α recovery')
ax.legend()
ax.grid(alpha=0.3)
ax.set_xlim(0, 0.75)
ax.set_ylim(0, 0.75)
plt.tight_layout()
plt.show()

for alpha_true, alpha_joint, alpha_profile in results:
    pass

## 7. Visual comparison: correction quality

Compare the corrected strong-pixel spectrum for:
- No correction (α=0)
- Default correction (α=0.2)
- Auto-detected correction (profile likelihood result)

In [ ]:
# Use a synthetic example so we KNOW the right answer: alpha_true = 0.35
alpha_true_demo = 0.35
pixel_demo = pixel_spec + alpha_true_demo * mean_c

_, _, alpha_auto_demo = profile_alpha(pixel_demo, mean_c, freqs, n_steps=80)

def esr_fit(spec, freqs):
    """Fit ESR14N to a spectrum, return (fitted_curve, params)."""
    def model(f, center, width, c0, c1, c2, offset):
        return esr14n(f, np.array([[center, width, c0, c1, c2, offset]])).squeeze()
    p0 = [freqs.mean(), 0.002, 0.02, 0.04, 0.02, 0.0]
    bounds = ([freqs.min(), 1e-4, 0, 0, 0, -0.2],
              [freqs.max(), 0.05, 0.5, 0.5, 0.5, 0.2])
    popt, _ = curve_fit(model, freqs, spec, p0=p0, bounds=bounds, maxfev=5000)
    return model(freqs, *popt), popt

scenarios = [
    (0.0,            'No correction (α=0)',         'gray'),
    (0.2,            'Default (α=0.2)',              'orange'),
    (alpha_auto_demo,f'Auto profile (α={alpha_auto_demo:.3f})', 'blue'),
    (alpha_true_demo,f'True (α={alpha_true_demo:.2f})',         'green'),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for alpha, label, color in scenarios:
    corrected = pixel_demo - alpha * mean_c
    fitted, _ = esr_fit(corrected, freqs)
    rms = np.sqrt(np.mean((corrected - fitted)**2))
    axes[0].plot(freqs, corrected, color=color, lw=1.5,
                 label=f'{label}  (fit RMS={rms:.2e})')

axes[0].set_xlabel('Frequency (GHz)')
axes[0].set_ylabel('ODMR contrast')
axes[0].set_title(f'Corrected spectra  (synthetic demo, α_true={alpha_true_demo})')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# Right: residuals after ESR fit
for alpha, label, color in scenarios:
    corrected = pixel_demo - alpha * mean_c
    fitted, _ = esr_fit(corrected, freqs)
    axes[1].plot(freqs, (corrected - fitted) * 1e3, color=color, lw=1.5, label=label)

axes[1].axhline(0, color='k', lw=0.8, ls='--')
axes[1].set_xlabel('Frequency (GHz)')
axes[1].set_ylabel('Residual (×10⁻³)')
axes[1].set_title('ESR fit residuals')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Apply to full image: spatial map of auto-detected α

Run profile likelihood on every pixel to see whether α varies spatially — 
if fluorescence is spatially uniform, α should be roughly constant.

In [ ]:
# Sub-sample: run on a 20×20 patch around the strong pixel
PATCH = 20
y0 = max(0, y_idx - PATCH // 2)
x0 = max(0, x_idx - PATCH // 2)
y1 = min(n_y, y0 + PATCH)
x1 = min(n_x, x0 + PATCH)

patch_data = data.data.isel(
    polarity=POL, freq_range=FR,
    y=slice(y0, y1), x=slice(x0, x1)
).values  # (patch_y, patch_x, n_freq)

patch_h, patch_w = patch_data.shape[:2]
alpha_map = np.full((patch_h, patch_w), np.nan)

n_steps_fast = 30  # coarse grid for speed
alphas_fast = np.linspace(0.0, 1.0, n_steps_fast)

for iy in range(patch_h):
    for ix in range(patch_w):
        spec = patch_data[iy, ix]
        chi2s = np.array([fit_esr_chi2(a, spec, mean_c, freqs) for a in alphas_fast])
        alpha_map[iy, ix] = alphas_fast[np.argmin(chi2s)]

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(alpha_map, cmap='RdBu_r', vmin=0, vmax=0.5)
plt.colorbar(im, ax=ax, label='Best α')
ax.set_title(f'Auto α map — {PATCH}×{PATCH} patch around strong pixel\n(pol={POL}, fr={FR})')
plt.tight_layout()
plt.show()


## 10. Re-investigation: where is the fluorescence hiding?

The strong pixel returned alpha=0. But the MIL sample **has** fluorescence.
Let us look at the raw data (before our `NormalizationProcessor`) to understand why.

### Key question
Does per-pixel normalization (divide by each pixel's max) absorb the fluorescence signal?


In [ ]:
# Reload WITHOUT our normalization — only spatial binning
data_raw_no_norm = data_binned

flat_raw = data_raw_no_norm.data.values.reshape(n_pol, n_fr, n_y * n_x, n_freq)
off_res_raw = np.concatenate([flat_raw[:,:,:,:4], flat_raw[:,:,:,-4:]], axis=-1).mean(axis=-1)
dip_raw = off_res_raw - flat_raw.min(axis=-1)

flat_norm2 = data.data.values.reshape(n_pol, n_fr, n_y * n_x, n_freq)
off_res_norm2 = np.concatenate([flat_norm2[:,:,:,:4], flat_norm2[:,:,:,-4:]], axis=-1).mean(axis=-1)
dip_norm2 = off_res_norm2 - flat_norm2.min(axis=-1)

r_raw = np.corrcoef(off_res_raw[0,1], dip_raw[0,1])[0,1]
r_norm = np.corrcoef(off_res_norm2[0,1], dip_norm2[0,1])[0,1]


In [ ]:
# Scatter: baseline vs dip depth, raw vs normalized
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(off_res_raw[0,1], dip_raw[0,1], alpha=0.05, s=1, c="blue")
axes[0].set_xlabel("Off-resonance baseline (raw)")
axes[0].set_ylabel("ESR dip depth (raw)")
axes[0].set_title(f"Raw MATLAB data   corr = {r_raw:.3f}")
axes[0].grid(alpha=0.3)

axes[1].scatter(off_res_norm2[0,1], dip_norm2[0,1], alpha=0.05, s=1, c="red")
axes[1].set_xlabel("Off-resonance baseline (normalized)")
axes[1].set_ylabel("ESR dip depth (normalized)")
axes[1].set_title(f"After NormalizationProcessor   corr = {r_norm:.3f}")
axes[1].grid(alpha=0.3)

plt.suptitle("Fluorescence signature: negative correlation persists after normalization")
plt.tight_layout()
plt.show()


## 11. Does the current correction pipeline move in the right direction?

The current pipeline: **normalize then correct**.
We test whether applying the correction (after normalization) reduces the correlation.


In [ ]:
_, mean_corr_norm2 = analyze_fluorescence_effects(data)

alpha_sweep = np.linspace(0, 0.5, 11)
corrs_post = []
for a in alpha_sweep:
    c_data = (data.data - a * mean_corr_norm2).values.reshape(n_pol, n_fr, n_y*n_x, n_freq)
    off_c = np.concatenate([c_data[:,:,:,:4], c_data[:,:,:,-4:]], axis=-1).mean(axis=-1)
    dip_c = off_c - c_data.min(axis=-1)
    r = np.corrcoef(off_c[0,1], dip_c[0,1])[0,1]
    corrs_post.append(r)
    direction = "WORSENING" if r < corrs_post[0] else "improving"



## 12. Multi-pixel strategy

As suggested: use top-N pixels, fit individually then with a shared alpha.
If fluorescence is global and uniform, all pixels should agree on the same alpha.


In [ ]:
# Select top-N pixels by deviation from spatial mean
flat_n = data.data.values.reshape(n_pol, n_fr, n_y * n_x, n_freq)
mean_sp = flat_n.mean(axis=2, keepdims=True)
deviations = np.sum((flat_n - mean_sp)**2, axis=-1)  # (pol, fr, n_pix)
sorted_idx = np.argsort(deviations[0, 1])[::-1]

N_PIXELS = 10
top_pixels = sorted_idx[:N_PIXELS]
mean_c_n = mean_corr_norm2.isel(polarity=0, freq_range=1).values
alphas_ms = np.linspace(0.0, 0.8, 41)

def esr_chi2(alpha, spec, freqs, mean_c):
    corrected = spec - alpha * mean_c
    def model(f, center, width, c0, c1, c2, offset):
        return esr14n(f, np.array([[center, width, c0, c1, c2, offset]])).squeeze()
    p0 = [freqs.mean(), 0.002, 0.02, 0.04, 0.02, 0.0]
    bounds = ([freqs.min(), 1e-4, 0, 0, 0, -0.2], [freqs.max(), 0.05, 0.5, 0.5, 0.5, 0.2])
    try:
        popt, _ = curve_fit(model, freqs, corrected, p0=p0, bounds=bounds, maxfev=5000)
        return float(np.sum((corrected - model(freqs, *popt))**2))
    except Exception:
        return np.inf

# Individual estimates
ind_alphas = []
for p in top_pixels:
    chi2s = [esr_chi2(a, flat_n[0,1,p], freqs, mean_c_n) for a in alphas_ms]
    ind_alphas.append(alphas_ms[np.argmin(chi2s)])

# Global: sum chi2 over all N pixels
global_chi2s = np.array([
    sum(esr_chi2(a, flat_n[0,1,p], freqs, mean_c_n) for p in top_pixels)
    for a in alphas_ms
])
alpha_global = alphas_ms[np.argmin(global_chi2s)]



## 13. The degeneracy argument

**Why does alpha=0 persist across all pixels and methods?**

The current mean-subtraction correction:
```
corrected(f) = pixel(f) - alpha * mean_corrected(f)
```
removes a component with the **spectral shape of the mean ODMR spectrum**.
This is designed to remove **ODMR-like cross-talk** (e.g. from a bulk NV background),
not a **flat non-resonant fluorescence background**.

For a **flat, uniform fluorescence** F (constant vs frequency):
```
raw(f)  = (1-alpha) * ESR(f; theta) + alpha
norm(f) = raw(f) / max(raw) = raw(f)   [max=1 by definition after normalization]
```
Fitting `(norm - alpha) / (1-alpha) = ESR(f)` is **degenerate**: the ESR contrast
and offset parameters absorb any alpha, giving the same chi2 for all alpha values.

**Consequence:**
- A globally uniform flat fluorescence is **undetectable** from ODMR data after
  per-pixel normalization — it needs an external reference measurement.
- The observed -0.24 residual correlation (after normalization) confirms **spatial
  variation** in the fluorescence, not a purely global contribution.
- The current correction formula targets ODMR-shaped cross-talk, not flat background.

**Revised recommendation for QEP-048:**
1. If fluorescence is spatially varying: apply correction **before normalization**
   and use correlation(baseline, dip_depth) as the minimization objective.
2. If fluorescence is truly global/uniform: needs an external reference or a
   different correction formula: `I_corrected = (I_norm - alpha) / (1 - alpha)`.
3. The multi-pixel approach improves robustness but does not resolve the degeneracy.


## 9. Summary and conclusions

In [ ]:
for alpha_true, alpha_joint, alpha_profile in results:
    pass